In [ ]:
%load_ext autoreload
%autoreload 2

Declaration of parameters (you must also add a tag for this cell - parameters)

In [ ]:
# specify parameters
pipeline_params={
}
step_params={
}
substep_params={   
    "threshold_iou" : 0.5,
    "threshold_MAp05": 0.01
}

In [ ]:
# define substep interface
from sinara.substep import NotebookSubstep, default_param_values, ENV_NAME, PIPELINE_NAME, ZONE_NAME, STEP_NAME, RUN_ID, ENTITY_NAME, ENTITY_PATH, SUBSTEP_NAME

substep = NotebookSubstep(pipeline_params, step_params, substep_params, **default_param_values("params/step_params.json"))

substep.interface(    
    tmp_inputs =
    [
        { ENTITY_NAME: "test_data" },
        { ENTITY_NAME: "predict_data" },
        { ENTITY_NAME: "test_config" }
    ]
)

substep.print_interface_info()

substep.exit_in_visualize_mode()

### Load ground truth and predicted test dataset markups

In [ ]:
import os.path as osp
import os
from utils.coco import load as coco_load
import json

tmp_inputs = substep.tmp_inputs()

# reading ground-true test dataset markup 
config_fn = osp.join(tmp_inputs.test_config, 'config.json')
with open(config_fn) as f_id:
    CONFIG = json.load(f_id)
eval_gt_file = osp.join(osp.join(tmp_inputs.test_data, CONFIG["test_coco_annotation"]))
eval_gt = coco_load(eval_gt_file)

# reading predicted test dataset markup 
eval_dt_file = osp.join(tmp_inputs.predict_data, 'eval_dt_torch.json')
eval_dt = coco_load(eval_dt_file)

iouType = eval_dt['info'].get('iouType', 'bbox')

### Evaluate the test dataset 

#### Function for building Precision-Recall Curve

In [ ]:
from utils.coco.encoder import NumpyEncoder
from pycocotools.cocoeval import COCOeval
from pycocotools.coco import COCO
from utils.extra.curves import Curves
import numpy as np

def reload_dict(_in_dict):
    return json.loads(json.dumps(_in_dict, cls=NumpyEncoder))

class Curves(Curves):
        
    def category_id_to_name(self, category_id):
        return self.cocoGt.cats[category_id + 1]['name']
    
    @staticmethod
    def calc_auc(recall_list, precision_list):
        mrec = recall_list
        mpre = precision_list

        for i in range(mpre.size - 1, 0, -1):
            mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])

        i = np.where(mrec[1:] != mrec[:-1])[0]
        return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])


    def build_curve(self, label):
            curve = []

            if self.useCats:
                cat_ids = list(range(self.eval['precision'].shape[2]))
            else:
                cat_ids = [0]

            for category_id in cat_ids:
                _label = f"{self.category_id_to_name(category_id)} "
                if len(cat_ids) == 1:
                    _label = ""

                precision_list = self.eval['precision'][:,
                                                        :, category_id, :, :].ravel()
                recall_list = self.recThrs
                scores = self.eval['scores'][:, :, category_id, :, :].ravel()
                auc = round(self.calc_auc(recall_list, precision_list), 4)

                curve.append(dict(
                    recall_list=recall_list,
                    precision_list=precision_list,
                    name=f'{_label}auc: {auc:.3f}',
                    scores=scores,
                    auc=auc,
                    category_id=category_id,
                ))

            return curve

#### Eval Precision-Recall Curve

In [ ]:
prepared_coco_in_dict = reload_dict(eval_gt)

_curves = []
prepared_anns = reload_dict(eval_dt['annotations'])

cocoGt = COCO(eval_gt_file)
cocoDt = cocoGt.loadRes(prepared_anns)

cur = Curves(
    cocoGt, 
    cocoDt, 
    iou_tresh=substep_params['threshold_iou'], 
    recall_count=1000, 
    iouType=iouType,
    # useCats=True,
)

_curve = cur.build_curve('category')
cur.plot_pre_rec(curves=_curve, plotly_backend=True)

#### Eval Average Precision, Recall Metrics

In [ ]:
from pycocotools.cocoeval import COCOeval

annType = eval_dt['info'].get('iouType', 'bbox')
imgIds=sorted(cocoGt.getImgIds())

# running evaluation
cocoEval = COCOeval(cocoGt,cocoDt,annType)
cocoEval.params.imgIds  = imgIds
cocoEval.evaluate()
cocoEval.accumulate()
cocoEval.summarize()

### Check model by metric mAP@0.5 IoU

In [ ]:
threshold_MAp05 = substep_params["threshold_MAp05"]

map_05 = cocoEval.stats[1]
print(f"mAP@0.5 = {map_05}")
assert map_05 > threshold_MAp05, f"The calculated MAp metric on the test dataset is less than the acceptable value <{threshold_MAp05}"